In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import json
import geopandas as gpd
from shapely.geometry import mapping    

# ── 1. Load your tables

In [2]:
coords_df = pd.read_csv("hospital_communities_with_coords.csv")
si_df     = pd.read_csv("hospital_si_results.csv")
edges_df  = pd.read_csv("agg_hospital_transfers.csv").copy()

# ── 2. Merge coords + SI results

In [3]:
hospitals = coords_df.merge(
    si_df[["hospital", "infection_frequency", "mean_epidemic_size",
           "mean_epidemic_size_pct", "mean_step_to_infection"]],
    on="hospital", how="left"
)

# ── 3. Export hospitals as JSON

In [4]:
hospitals_out = hospitals[[
    "hospital", "Name", "Postcode", "latitude", "longitude",
    "community_id", "weighted_in_degree", "weighted_out_degree",
    "total_transfers", "cross_community_bridges", "node_size",
    "infection_frequency", "mean_epidemic_size",
    "mean_epidemic_size_pct", "mean_step_to_infection"
]].copy()
hospitals_out.columns = [
    "id", "name", "postcode", "lat", "lon",
    "community_id", "weighted_in_degree", "weighted_out_degree",
    "total_transfers", "cross_community_bridges", "node_size",
    "infection_frequency", "mean_epidemic_size",
    "mean_epidemic_size_pct", "mean_step_to_infection"
]
hospitals_out.to_json("si_hospitals.json", orient="records", indent=2)
print(f"✓ Hospitals exported: {len(hospitals_out)} rows")

✓ Hospitals exported: 125 rows


# ── 4. Export edges as JSON 

In [5]:
edges_out = edges_df[edges_df["weight"] >= 100][[
    "source", "target", "weight"
]].copy()
edges_out.to_json("si_edges.json", orient="records", indent=2)
print(f"✓ Edges exported: {len(edges_out)} rows")

✓ Edges exported: 928 rows


# ── 5. Export ICB boundaries as GeoJSON

In [6]:
icb = gpd.read_file("icb_boundaries.geojson")
icb_out = icb[["ICB23CD", "ICB23NM", "LAT", "LONG", "geometry"]].copy()
icb_out.columns = ["icb_code", "icb_name", "lat", "lon", "geometry"]
icb_out.to_file("si_icb_boundaries.geojson", driver="GeoJSON")
print(f"✓ ICB boundaries exported: {len(icb_out)} ICBs")

✓ ICB boundaries exported: 42 ICBs


# ── 6. Confirm files

In [ ]:
import os
for f in ["si_hospitals.json", "si_edges.json", "si_icb_boundaries.geojson"]:
    size = os.path.getsize(f"{f}")
    print(f"  {f}: {size/1024:.1f} KB")

print("\n✓ All done. Download these 3 files from to HES_data.")

  si_hospitals.json: 56.3 KB
  si_edges.json: 60.1 KB
  si_icb_boundaries.geojson: 7654.6 KB

✓ All done. Download these 3 files from /dbfs/tmp/ to your desktop.
